In [ ]:
import pandas as pd
import scanpy as sc
import anndata as ad

### Load SpinMCIA Results

Make sure to change the paths and the .csv file names depending on usage

In [ ]:
base_path = ""

# Load the CSV files into DataFrames
hic_block_scores = pd.read_csv(base_path + "block_scores_HiC.csv", header=None, skiprows=1) # 
dna_met_block_scores = pd.read_csv(base_path + "block_scores_SecondModality.csv", header=None, skiprows=1) 
global_scores = pd.read_csv(base_path + "global_scores.csv",header=None,skiprows=1)

hic_block_loadings = pd.read_csv(base_path + "block_loadings_HiC.csv")
dna_met_block_loadings = pd.read_csv(base_path + "block_loadings_SecondModality.csv")
eigenvalues = pd.read_csv(base_path + "eigvals.csv")

block_score_weights = pd.read_csv(base_path + "block_score_weights.csv")

In [ ]:
global_scores

### Drop First Column only if needed (if you see cell_0, cell_1, etc..)

In [ ]:
# global_scores = global_scores.iloc[:, 1:]

In [ ]:
adata = ad.AnnData(global_scores)

In [ ]:
# Load the cell type information and ensure it's the correct length
cell_type_information = pd.read_csv("", usecols=[0])

# Convert the single column DataFrame to a Series
cell_type_information = cell_type_information.iloc[:, 0]

# Make sure the length of the cell_type_information matches the number of observations in adata
if len(cell_type_information) == adata.shape[0]:
    # Assign the cell types to adata.obs with matching index
    adata.obs["cell_type"] = pd.Categorical(cell_type_information)
else:
    print(f"Length mismatch: {len(cell_type_information)} in cell_type_information vs {adata.shape[0]} in adata")

adata.obs


In [ ]:
cell_type_information

In [ ]:
adata

In [ ]:
print(adata.X)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=250)  # Adjust n_neighbors and n_pcs as needed

In [ ]:
# Compute UMAP
sc.tl.umap(adata)

In [ ]:
adata

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6,6), dpi=600)

sc.pl.umap(
    adata,
    color='cell_type',
    title='scHi-C + DNA Methylation',     # no title
    ax=ax,
    show=False
)

### Metrics

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.metrics import (
    adjusted_rand_score, adjusted_mutual_info_score, normalized_mutual_info_score,
    accuracy_score, silhouette_score, davies_bouldin_score, calinski_harabasz_score
)
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
import scib_metrics
from tqdm.notebook import tqdm # For a nice progress bar during the sweep

# --- HELPER FUNCTION ---
def map_clusters_to_ground_truth(y_true, y_pred):
    ct = pd.crosstab(y_pred, y_true)
    mapping_dict = ct.idxmax(axis=1).to_dict()
    return np.array([mapping_dict.get(val, "Unmapped") for val in y_pred])

# --- PREPARE DATA ---
X = adata.X
X_umap_2d = adata.obsm['X_umap']
y_true_str = adata.obs['cell_type'].astype(str).values
y_int = pd.Categorical(y_true_str).codes
n_cells = X.shape[0]

# --- 1. SPATIAL METRICS ---
print("Calculating Spatial Metrics (Silhouette, DBI, CHI)...")
sil_emb  = silhouette_score(X, y_true_str)
sil_umap = silhouette_score(X_umap_2d, y_true_str)
dbi      = davies_bouldin_score(X, y_true_str)
chi      = calinski_harabasz_score(X, y_true_str)

# --- 2. EXACT HYPERPARAMETER SWEEP ---
print("Running Rigorous Hyperparameter Sweep for Best & Average Metrics...")
sqrt_n = int(np.sqrt(n_cells))
min_nn, max_nn = max(10, sqrt_n // 2), min(n_cells - 1, sqrt_n * 2)
n_neighbors_list = np.unique(np.linspace(min_nn, max_nn, num=10, dtype=int))
resolutions = np.arange(0.1, 5.1, 0.1)

best_ari = -1.0
best_metrics = {}

# 🌟 NEW: List to track all configurations for the Top 5 Averages
all_sweep_results = []

# Sweeping through the grid
for nn in tqdm(n_neighbors_list, desc="Neighbors Sweep"):
    sc.pp.neighbors(adata, n_neighbors=nn, use_rep='X', metric='euclidean')
    for res in resolutions:
        key = f"leiden_nn{nn}_r{res:.1f}"
        sc.tl.leiden(adata, resolution=res, key_added=key)
        
        y_pred = adata.obs[key].values
        y_pred_mapped = map_clusters_to_ground_truth(y_true_str, y_pred)
        
        # Calculate current metrics
        current_ari = adjusted_rand_score(y_true_str, y_pred_mapped)
        current_nmi = normalized_mutual_info_score(y_true_str, y_pred_mapped)
        current_ami = adjusted_mutual_info_score(y_true_str, y_pred_mapped)
        
        current_raw_ari = adjusted_rand_score(y_true_str, y_pred)
        current_raw_nmi = normalized_mutual_info_score(y_true_str, y_pred)
        current_raw_ami = adjusted_mutual_info_score(y_true_str, y_pred)
        
        # 🌟 NEW: Save to our tracking list
        all_sweep_results.append({
            'ARI': current_ari,
            'NMI': current_nmi,
            'AMI': current_ami,
            'Raw_ARI': current_raw_ari,
            'Raw_NMI': current_raw_nmi,
            'Raw_AMI': current_raw_ami
        })
        
        # Keep the metrics from the sweep that produces the highest ARI
        if current_ari > best_ari:
            best_ari = current_ari
            best_metrics = {
                "Best n_neighbors": nn,
                "Best resolution": res,
                "ARI": current_ari,
                "NMI": current_nmi,
                "AMI": current_ami,
                "Accuracy": accuracy_score(y_true_str, y_pred_mapped)
            }

# 🌟 NEW: Calculate the Top 5 Averages using Pandas
df_sweep = pd.DataFrame(all_sweep_results)
avg_ari = df_sweep['ARI'].nlargest(5).mean()
avg_nmi = df_sweep['NMI'].nlargest(5).mean()
avg_ami = df_sweep['AMI'].nlargest(5).mean()

avg_raw_ari = df_sweep['Raw_ARI'].nlargest(5).mean()
avg_raw_nmi = df_sweep['Raw_NMI'].nlargest(5).mean()
avg_raw_ami = df_sweep['Raw_AMI'].nlargest(5).mean()

# --- 3. SCIB METRICS ---
print("Calculating SCIB Metrics (ASW & cLISI)...")
adata_3d = adata.copy()
sc.tl.umap(adata_3d, n_components=3)
raw_asw = float(silhouette_score(adata_3d.obsm['X_umap'], y_int))
asw_score = (raw_asw + 1.0) / 2.0

nbrs = NearestNeighbors(n_neighbors=50, metric='euclidean').fit(X)
distances, indices = nbrs.kneighbors(X)
distances = np.maximum(distances, 1e-12)

row = np.repeat(np.arange(n_cells), 50)
col = indices.flatten()
dist_mat = csr_matrix((distances.flatten(), (row, col)), shape=(n_cells, n_cells))
clisi_score = float(scib_metrics.clisi_knn(dist_mat, y_int))

# --- SUMMARY DATAFRAME ---
final_results = {
    "ARI": best_metrics["ARI"],
    "NMI": best_metrics["NMI"],
    "AMI": best_metrics["AMI"],
    "Average ARI (Top 5)": avg_ari,     
    "Average NMI (Top 5)": avg_nmi,     
    "Average AMI (Top 5)": avg_ami,               
    "SCIB: Cell-Type ASW": asw_score,
    "SCIB: Graph cLISI": clisi_score,
    "Sweep Config: n_neighbors": best_metrics["Best n_neighbors"],
    "Sweep Config: resolution": best_metrics["Best resolution"]
}

df_metrics = pd.DataFrame(list(final_results.items()), columns=["Metric", "Score"])
df_metrics.set_index("Metric", inplace=True)

print("\n✅ Metrics Generation Complete!")
display(df_metrics)

### Downstream Analysis

Please refer too the Lee 200kb tutorials to get a sense of the downstream analysis